# Task 3 — Gender × occasion/usage

Task 3 asks for models that predict **who the item is for** (`gender`) and **what occasion/usage it suits** (`usage` in the CSV).

This notebook measures how tightly those two labels are coupled, then compares:

| Design | What you train | What you predict |
|---|---|---|
| **A. One model, two heads** | 1 network, 2 output heads | `gender` and `occasion` in one forward pass |
| **A+. Hybrid (image + metadata)** | CNN image vector + ANN metadata vector, then 2 heads | same two labels |
| **B. Two separate models** | 1 CNN for gender + 1 CNN for occasion | same labels, double cost |
| **C. One combined class** | 1 softmax over `gender \| occasion` | a single joint tag |

**Two heads = one model, two prediction types** (not two separate networks).

Data: `data/FashionDataset/train/styles_train.csv`. Test `gender` / `usage` / other metadata cells are empty.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "eda":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "FashionDataset" / "train" / "styles_train.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set2

print("data:", DATA_PATH)
print("exists:", DATA_PATH.exists())


data: /Users/nhan.ngo/rmit/COSC2753-Project/data/FashionDataset/train/styles_train.csv
exists: True


## Load and clean


In [2]:
raw = pd.read_csv(DATA_PATH)
unnamed = [c for c in raw.columns if str(c).startswith("Unnamed")]
df = raw.drop(columns=unnamed).rename(columns={"usage": "occasion"})
df["gender"] = df["gender"].fillna("Missing")
df["occasion"] = df["occasion"].fillna("Missing")
df["combined"] = df["gender"] + " | " + df["occasion"]

print("rows:", len(df))
print("gender levels:", df["gender"].nunique(), sorted(df["gender"].unique()))
print("occasion levels:", df["occasion"].nunique(), sorted(df["occasion"].unique()))
print("observed combined classes:", df["combined"].nunique())
print("cartesian product:", df["gender"].nunique() * df["occasion"].nunique())
df[["id", "gender", "occasion", "combined"]].head()


rows: 38617
gender levels: 5 ['Boys', 'Girls', 'Men', 'Unisex', 'Women']
occasion levels: 9 ['Casual', 'Ethnic', 'Formal', 'Home', 'Missing', 'Party', 'Smart Casual', 'Sports', 'Travel']
observed combined classes: 27
cartesian product: 45


,id,gender,occasion,combined
0,1163,Men,Sports,Men | Sports
1,1164,Men,Sports,Men | Sports
2,1165,Men,Sports,Men | Sports
3,1525,Unisex,Casual,Unisex | Casual
4,1526,Unisex,Sports,Unisex | Sports


## Univariate: each target on its own

If either label is already badly imbalanced, combining them will make the tail worse.


In [3]:
def count_table(col: str) -> pd.DataFrame:
    out = df[col].value_counts().rename("n").reset_index()
    out.columns = [col, "n"]
    out["pct"] = (100 * out["n"] / out["n"].sum()).round(2)
    return out


gender_counts = count_table("gender")
occasion_counts = count_table("occasion")
display(Markdown("**gender**"))
display(gender_counts)
display(Markdown("**occasion** (`usage`)"))
display(occasion_counts)

fig = px.bar(
    gender_counts,
    x="n",
    y="gender",
    orientation="h",
    text="pct",
    title="Train images by gender",
    labels={"n": "Number of images", "gender": "gender"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

fig = px.bar(
    occasion_counts,
    x="n",
    y="occasion",
    orientation="h",
    text="pct",
    title="Train images by occasion / usage",
    labels={"n": "Number of images", "occasion": "occasion"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


**gender**

,gender,n,pct
0,Men,20918,54.17
1,Women,14160,36.67
2,Unisex,2080,5.39
3,Boys,814,2.11
4,Girls,645,1.67


**occasion** (`usage`)

,occasion,n,pct
0,Casual,29641,76.76
1,Sports,3940,10.20
2,Ethnic,2570,6.66
3,Formal,2300,5.96
4,Missing,72,0.19
5,Smart Casual,55,0.14
6,Travel,25,0.06
7,Party,13,0.03
8,Home,1,0.00


## Joint distribution

Counts, **% of occasion within each gender**, and **% of gender within each occasion**.


In [4]:
counts = pd.crosstab(df["gender"], df["occasion"], margins=True)
row_pct = pd.crosstab(df["gender"], df["occasion"], normalize="index").mul(100).round(1)
col_pct = pd.crosstab(df["gender"], df["occasion"], normalize="columns").mul(100).round(1)

display(Markdown("**Counts**"))
display(counts)
display(Markdown("**% of occasion within gender** (rows sum to 100)"))
display(row_pct)
display(Markdown("**% of gender within occasion** (columns sum to 100)"))
display(col_pct)

long = df.groupby(["gender", "occasion"], observed=True).size().reset_index(name="n")
long["pct_within_gender"] = (
    100 * long["n"] / long.groupby("gender")["n"].transform("sum")
).round(2)

gender_order = gender_counts["gender"].tolist()
fig = px.bar(
    long,
    x="gender",
    y="pct_within_gender",
    color="occasion",
    barmode="stack",
    category_orders={"gender": gender_order},
    title="Occasion mix within each gender",
    labels={
        "pct_within_gender": "% within gender",
        "gender": "gender",
        "occasion": "occasion",
    },
)
fig.show()

fig = px.bar(
    long,
    x="occasion",
    y="n",
    color="gender",
    barmode="stack",
    title="Gender mix within each occasion (counts)",
    labels={"n": "Number of images", "occasion": "occasion", "gender": "gender"},
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

heat = pd.crosstab(df["gender"], df["occasion"])
fig = px.imshow(
    heat,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="gender x occasion counts",
    labels={"color": "Count"},
    height=420,
)
fig.show()

fig = px.imshow(
    row_pct,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="Teal",
    title="Occasion mix within gender (%)",
    labels={"color": "% within gender"},
    height=420,
)
fig.show()


**Counts**

occasion,Casual,Ethnic,Formal,Home,Missing,Party,Smart Casual,Sports,Travel,All
gender,,,,,,,,,,
Boys,783,10,0,0,0,0,0,21,0,814
Girls,635,8,0,0,0,0,0,2,0,645
Men,15691,90,2193,0,23,0,44,2876,1,20918
Unisex,1758,0,1,1,18,0,0,282,20,2080
Women,10774,2462,106,0,31,13,11,759,4,14160
All,29641,2570,2300,1,72,13,55,3940,25,38617


**% of occasion within gender** (rows sum to 100)

occasion,Casual,Ethnic,Formal,Home,Missing,Party,Smart Casual,Sports,Travel
gender,,,,,,,,,
Boys,96.2,1.2,0.0,0.0,0.0,0.0,0.0,2.6,0.0
Girls,98.4,1.2,0.0,0.0,0.0,0.0,0.0,0.3,0.0
Men,75.0,0.4,10.5,0.0,0.1,0.0,0.2,13.7,0.0
Unisex,84.5,0.0,0.0,0.0,0.9,0.0,0.0,13.6,1.0
Women,76.1,17.4,0.7,0.0,0.2,0.1,0.1,5.4,0.0


**% of gender within occasion** (columns sum to 100)

occasion,Casual,Ethnic,Formal,Home,Missing,Party,Smart Casual,Sports,Travel
gender,,,,,,,,,
Boys,2.6,0.4,0.0,0.0,0.0,0.0,0.0,0.5,0.0
Girls,2.1,0.3,0.0,0.0,0.0,0.0,0.0,0.1,0.0
Men,52.9,3.5,95.3,0.0,31.9,0.0,80.0,73.0,4.0
Unisex,5.9,0.0,0.0,100.0,25.0,0.0,0.0,7.2,80.0
Women,36.3,95.8,4.6,0.0,43.1,100.0,20.0,19.3,16.0


## How dependent are the labels?

Cramer's V is 0 when `gender` and `occasion` are independent, and 1 when one fully determines the other.


In [5]:
def cramers_v(a: pd.Series, b: pd.Series) -> tuple[float, float, int]:
    tab = pd.crosstab(a, b).to_numpy(dtype=float)
    n = tab.sum()
    expected = tab.sum(axis=1, keepdims=True) * tab.sum(axis=0, keepdims=True) / n
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((tab - expected) ** 2 / np.where(expected == 0, np.nan, expected))
    r, k = tab.shape
    phi2 = chi2 / n
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return float(np.sqrt(phi2corr / denom)), float(chi2), int(n)


v, chi2, n = cramers_v(df["gender"], df["occasion"])

ct = pd.crosstab(df["gender"], df["occasion"]).to_numpy(dtype=float)
p = ct / ct.sum()
pg = p.sum(axis=1, keepdims=True)
pu = p.sum(axis=0, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    mi = float(np.nansum(np.where(p > 0, p * np.log(p / (pg * pu)), 0.0)))
    hg = float(-np.nansum(np.where(pg > 0, pg * np.log(pg), 0.0)))
    hu = float(-np.nansum(np.where(pu > 0, pu * np.log(pu), 0.0)))
nmi = mi / min(hg, hu) if min(hg, hu) > 0 else np.nan

assoc = pd.DataFrame(
    {
        "metric": [
            "Cramer's V",
            "Mutual information (bits)",
            "Normalised MI (MI / min(H_gender, H_occasion))",
            "Gender entropy (bits)",
            "Occasion entropy (bits)",
        ],
        "value": [
            v,
            mi / np.log(2),
            nmi,
            hg / np.log(2),
            hu / np.log(2),
        ],
    }
).round(4)
display(assoc)

msg = f"""
**Reading:** Cramer's V = **{v:.3f}** is a *moderate* association. The labels are **not independent**,
but one does **not** determine the other.

Notable couplings (from the heatmaps):

- **Ethnic** is almost entirely **Women** (~96%).
- **Formal** is almost entirely **Men** (~95%).
- **Boys / Girls** are almost entirely **Casual**.
- **Casual** (77% of the data) is mixed across Men and Women, so a combined class is not a shortcut there.
"""
display(Markdown(msg))


,metric,value
0,Cramer's V,0.2071
1,Mutual information (bits),0.1368
2,"Normalised MI (MI / min(H_gender, H_occasion))",0.1166
3,Gender entropy (bits),1.4528
4,Occasion entropy (bits),1.1730



**Reading:** Cramer's V = **0.207** is a *moderate* association. The labels are **not independent**,
but one does **not** determine the other.

Notable couplings (from the heatmaps):

- **Ethnic** is almost entirely **Women** (~96%).
- **Formal** is almost entirely **Men** (~95%).
- **Boys / Girls** are almost entirely **Casual**.
- **Casual** (77% of the data) is mixed across Men and Women, so a combined class is not a shortcut there.


## Combined class: `gender | occasion`

A single target would have one class per observed pair. Compare class count and the long tail against the two separate targets.


In [6]:
combined_counts = df["combined"].value_counts().rename("n").reset_index()
combined_counts.columns = ["combined", "n"]
combined_counts["pct"] = (100 * combined_counts["n"] / combined_counts["n"].sum()).round(2)
display(combined_counts)

fig = px.bar(
    combined_counts,
    x="n",
    y="combined",
    orientation="h",
    text="n",
    title="Combined class counts (gender | occasion)",
    labels={"n": "Number of images", "combined": "combined class"},
    height=max(480, 18 * len(combined_counts) + 80),
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


def tail_summary(series: pd.Series, name: str) -> pd.DataFrame:
    vc = series.value_counts()
    rows = []
    for thr in [1, 5, 10, 20, 50, 100]:
        mask = vc < thr
        rows.append(
            {
                "target": name,
                "min_examples": thr,
                "n_classes": int(len(vc)),
                "classes_below": int(mask.sum()),
                "images_below": int(vc[mask].sum()),
                "pct_images_below": round(100 * vc[mask].sum() / vc.sum(), 2),
            }
        )
    return pd.DataFrame(rows)


imbalance = pd.concat(
    [
        tail_summary(df["gender"], "gender (independent)"),
        tail_summary(df["occasion"], "occasion (independent)"),
        tail_summary(df["combined"], "combined (gender | occasion)"),
    ],
    ignore_index=True,
)
display(Markdown("**Classes with fewer than k images**"))
display(imbalance)

empty_pairs = df["gender"].nunique() * df["occasion"].nunique() - df["combined"].nunique()
print(
    f"Possible pairs: {df['gender'].nunique() * df['occasion'].nunique()}  |  "
    f"observed: {df['combined'].nunique()}  |  empty cells: {empty_pairs}"
)


,combined,n,pct
0,Men | Casual,15691,40.63
1,Women | Casual,10774,27.90
2,Men | Sports,2876,7.45
3,Women | Ethnic,2462,6.38
4,Men | Formal,2193,5.68
5,Unisex | Casual,1758,4.55
6,Boys | Casual,783,2.03
7,Women | Sports,759,1.97
8,Girls | Casual,635,1.64
9,Unisex | Sports,282,0.73


**Classes with fewer than k images**

,target,min_examples,n_classes,classes_below,images_below,pct_images_below
0,gender (independent),1,5,0,0,0.00
1,gender (independent),5,5,0,0,0.00
2,gender (independent),10,5,0,0,0.00
3,gender (independent),20,5,0,0,0.00
4,gender (independent),50,5,0,0,0.00
5,gender (independent),100,5,0,0,0.00
6,occasion (independent),1,9,0,0,0.00
7,occasion (independent),5,9,1,1,0.00
8,occasion (independent),10,9,1,1,0.00
9,occasion (independent),20,9,2,14,0.04


Possible pairs: 45  |  observed: 27  |  empty cells: 18


## Hybrid architecture (recommended build)

Yes: **one model**, not two. Flow:

1. **Image branch (CNN)** — image → conv blocks → feature map → flatten → vector `v_img`
2. **Metadata branch (ANN)** — selected CSV fields → embeddings / one-hot → small MLP → vector `v_meta`
3. **Fusion** — `v = concat(v_img, v_meta)` (optional extra MLP on `v`)
4. **Two heads (two small ANNs)** — Head 1: `v → gender`. Head 2: `v → occasion`

```text
  image ──► CNN ──► flatten ──► v_img ──┐
                                        ├─► concat ──► v ──┬─► ANN head ──► gender
  CSV  ──► embed/MLP ─────────► v_meta ─┘                  └─► ANN head ──► occasion
```

Do **not** feed `gender` or `occasion`/`usage` into the metadata branch — those are the targets (label leakage).

Safe CSV fields: `masterCategory`, `subCategory`, `articleType`, `baseColour`, `season`, `year`.  
Unsafe: `id`, `productDisplayName`, `gender`, `usage`.

**Test-time catch:** `styles_prediction.csv` leaves metadata blank. So either:

- **Image-only at test** (drop the metadata branch, or train with metadata dropout), or
- **Pipeline from Task 1:** predicted `articleType` (and only that) into `v_meta`. Errors from Task 1 will flow into Task 3.


In [ ]:

META_SAFE = [
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "year",
]
META_UNSAFE = ["id", "productDisplayName", "gender", "occasion"]

rows = []
for col in META_SAFE:
    v_g, _, _ = cramers_v(df["gender"], df[col].fillna("Missing").astype(str))
    v_o, _, _ = cramers_v(df["occasion"], df[col].fillna("Missing").astype(str))
    rows.append(
        {
            "metadata_field": col,
            "Cramers_V_vs_gender": round(v_g, 3),
            "Cramers_V_vs_occasion": round(v_o, 3),
            "n_unique": int(df[col].nunique(dropna=True)),
        }
    )
meta_assoc = pd.DataFrame(rows).sort_values("Cramers_V_vs_occasion", ascending=False)
display(Markdown("**How useful is each CSV field as metadata?** (do not use gender/occasion themselves)"))
display(meta_assoc)

fig = px.bar(
    meta_assoc.melt(
        id_vars="metadata_field",
        value_vars=["Cramers_V_vs_gender", "Cramers_V_vs_occasion"],
        var_name="target",
        value_name="Cramers_V",
    ),
    x="metadata_field",
    y="Cramers_V",
    color="target",
    barmode="group",
    title="Metadata vs targets (Cramer's V) — hybrid input ranking",
    labels={"metadata_field": "CSV field", "Cramers_V": "Cramer's V", "target": "Target"},
)
fig.update_layout(xaxis_tickangle=-20)
fig.show()


## Decision

**Two heads = one model, two prediction types.**

The hybrid (CNN image vector + metadata ANN vector → concat → two ANN heads) is still design **A**, with an extra input branch. It is not two models.


In [ ]:

rare_occasion = occasion_counts.loc[occasion_counts["n"] < 100, "occasion"].tolist()
rare_combined = combined_counts.loc[combined_counts["n"] < 50, "combined"].tolist()
rare_occ_txt = ", ".join(rare_occasion) if rare_occasion else "none"
top_meta = ", ".join(meta_assoc["metadata_field"].head(3).tolist())

msg = f"""
### Recommendation: **one hybrid model, two heads**

```
image → CNN → flatten → v_img
CSV (not gender/usage) → ANN → v_meta
v = concat(v_img, v_meta)
     ├─ ANN head → gender   ({df["gender"].nunique()} classes)
     └─ ANN head → occasion ({df["occasion"].nunique()} classes)
Loss = L_gender + L_occasion
```

| Design | Models | Inputs | Verdict |
|---|---:|---|---|
| **A+. Hybrid + two heads** | **1** | image + safe CSV | **Preferred if metadata exists at test** |
| **A. Image-only two heads** | 1 | image | Fallback (test CSV is empty) |
| **B. Two separate models** | 2 | image (x2) | Duplicates the CNN; ignore |
| **C. Combined class** | 1 | image | Avoid ({len(rare_combined)} joint classes with < 50 images) |

**Answers**

- *Two heads* = **one CNN/hybrid**, two outputs — **not** two separate models.
- Metadata ANN is a **second input branch**, not a second model. Both vectors are fused, then both heads read the fused vector.
- Strongest safe metadata vs these targets (this run): **{top_meta}**.
- Never put `gender` / `usage` into `v_meta`.

**Occasion cleanup** (both heads still apply): drop or merge rare values {rare_occ_txt}.
"""
display(Markdown(msg))
